# 🛡️TIF Análisis de Detección de Intrusiones en Red

**Grupo 1**: Gonza Gabriela · Casasola Hernán · Biazutti Luciano · Lera Aníbal Iván · Alvarado Marcelo

**Módulo**: Ciencias de Datos y Optimización de Modelos

**Carrera**: Tecnicatura Universitaria en Ciencias de Datos e IA Aplicada — UPATECO

---

#  Generación Dataset con submuestreo


##  Librerías

In [1]:
import pandas as pd
import hashlib as hl
import pyarrow
import io
import requests

import numpy as np

from pandas.api.types import is_string_dtype

from sklearn.ensemble import IsolationForest
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE


#  Funciones


In [2]:
#Esta función carga un dataset desde el repo de Github
def cargarDataset(archivos):

  GITHUB_USER = 'jotaeleb'
  GITHUB_REPO = 'tif-ciencias-de-datos'
  GITHUB_BRANCH = 'main'
  DATASET_DIR = 'dataset'

  BASE_URL = (
    f'https://raw.githubusercontent.com/'
    f'{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}/{DATASET_DIR}'
  )

  partes = []
  for archivo in archivos:
    url = f'{BASE_URL}/{archivo}'
    print(f"Cargando archivo {url}")
    partes.append(pd.read_parquet(io.BytesIO(requests.get(url).content)))


  completo = pd.concat(partes,ignore_index=True)
  completo['Label'] = completo['Label'].str.replace(r'[^\w\s-]', '-', regex=True)

  return completo

#Esta función devuelve las columnas numéricas del dataset
def colNumericas(dFrame):
  columnas = []
  for columna, tipoDato in dFrame.dtypes.items():
    if not is_string_dtype(tipoDato):
      columnas.append(columna)
  return columnas

##  Carga dataset EDA

*   2.824.951 registros
*   66 columnas


In [3]:
print("--> Carga Dataset <--")
df = cargarDataset(["dataset_limpio_parte_1.parquet","dataset_limpio_parte_2.parquet","dataset_limpio_parte_3.parquet","dataset_limpio_parte_4.parquet"])

print("--> shape <--", df.shape)

print("--> Cantidad de instancias variable objetivo <--")
print(df['Label'].value_counts())


--> Carga Dataset <--
Cargando archivo https://raw.githubusercontent.com/jotaeleb/tif-ciencias-de-datos/main/dataset/dataset_limpio_parte_1.parquet
Cargando archivo https://raw.githubusercontent.com/jotaeleb/tif-ciencias-de-datos/main/dataset/dataset_limpio_parte_2.parquet
Cargando archivo https://raw.githubusercontent.com/jotaeleb/tif-ciencias-de-datos/main/dataset/dataset_limpio_parte_3.parquet
Cargando archivo https://raw.githubusercontent.com/jotaeleb/tif-ciencias-de-datos/main/dataset/dataset_limpio_parte_4.parquet
--> shape <-- (2824951, 66)
--> Cantidad de instancias variable objetivo <--
Label
BENIGN         2268589
DoS-DDoS        379554
PortScan        158804
Brute Force      13826
Web Attack        2159
Botnet            1956
Rare Attack         63
Name: count, dtype: int64


##  Remuestreo

In [4]:
#Obtener listado de las columnas numéricas del dataset
columnas = colNumericas(df)

#Asignar a X las columnas numéricas del dataset
X = df[columnas]

#Asignar a etiqueta (Y) la variable objetivo
etiqueta = df["Label"]

remuestreo = RandomUnderSampler(
  sampling_strategy={
      'BENIGN': 50000,      # reducir de 2.27M a 50k
      'DoS-DDoS': 50000,    # reducir de 380k a 50k
      'PortScan': 50000,    # reducir de 180k a 50k
      'Brute Force': 13826,  # mantener
      'Web Attack': 2159,    # mantener
      'Botnet': 1956,        # mantener
      'Rare Attack': 63      # mantener
  }, random_state=42)

X_remuestreo, etiqueta_remuestreo = remuestreo.fit_resample(X, etiqueta)

print(len(X_remuestreo),len(etiqueta_remuestreo))

print(etiqueta_remuestreo.value_counts())


168004 168004
Label
BENIGN         50000
PortScan       50000
DoS-DDoS       50000
Brute Force    13826
Web Attack      2159
Botnet          1956
Rare Attack       63
Name: count, dtype: int64


##  Grabar dataset con submuestreo

In [5]:
df = pd.concat([X_remuestreo, etiqueta_remuestreo], axis=1)

df.to_parquet("dataset_limpio_submuestreo.parquet", compression='gzip', index=False)